# Chapter 5 – Reliability & Probabilistic Assessment

**Project:** Floating Submerged Tunnel – Helsinki to Tallinn  
**Objective:** Compute the structural reliability index β and annual failure probability P_f for governing limit states.

---

## 5.1 Limit States

| LS | Type | Description | Target β (50-yr) |
|----|------|-------------|-------------------|
| LS1 | ULS – Tether tension | Tether yield under extreme loading | 4.7 |
| LS2 | ULS – Tether fatigue | Accumulated fatigue damage > 1 | 3.3 |
| LS3 | SLS – Heave motion | Excessive vertical displacement | 2.5 |
| LS4 | ALS – Tether failure | Residual strength after one tether loss | 3.1 |

*Target values from DNVGL-OS-C101 / ISO 2394.*


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats, optimize

%matplotlib inline
plt.rcParams['figure.dpi'] = 120

rng = np.random.default_rng(2024)

## 5.2 Random Variables

| Variable | Symbol | Distribution | Mean | CoV | Unit |
|----------|--------|--------------|------|-----|------|
| Tether yield strength | R | Lognormal | 355 | 0.07 | MPa |
| Design load (tether tension) | S | Lognormal | *(EVA)* | 0.15 | kN |
| Tether area | A | Normal | *(design)* | 0.03 | m² |
| Drag coefficient | C_D | Normal | 1.0 | 0.20 | – |
| Wave height (100-yr) | H_s | Gumbel | *(EVA)* | 0.10 | m |


In [ ]:
# -----------------------------------------------------------------------
# LS1 – Tether tension limit state
# g(R, S) = R·A - S  > 0  (safe)
# R: tether yield strength [MPa → kN/m²]
# A: tether cross-sectional area [m²]
# S: tether design tension [kN]
# -----------------------------------------------------------------------

# --- Random variable parameters ---
# Resistance: R [kN] ~ Lognormal
mu_R   = 355e3   # mean yield stress [kN/m²]
cov_R  = 0.07
A_mean = np.pi * (0.150)**2 / 4   # mean tether area [m²] – update from Ch3
cov_A  = 0.03

# Capacity = R * A  (lognormal * normal ≈ lognormal for small CoV)
mu_Rc  = mu_R * A_mean   # [kN]
cov_Rc = np.sqrt(cov_R**2 + cov_A**2)
sigma_Rc = mu_Rc * cov_Rc

# Load: S [kN] ~ Lognormal – placeholder until Ch3/Ch4 values available
mu_S   = 1200.0   # TODO: replace with Chapter 3 design tension [kN]
cov_S  = 0.15
sigma_S = mu_S * cov_S

print("=== LS1 Random Variables ===")
print(f"Capacity  R·A: mean = {mu_Rc:.0f} kN,  σ = {sigma_Rc:.0f} kN,  CoV = {cov_Rc:.3f}")
print(f"Load      S:   mean = {mu_S:.0f} kN,  σ = {sigma_S:.0f} kN,  CoV = {cov_S:.3f}")

## 5.3 First-Order Reliability Method (FORM)

For a lognormal capacity R and lognormal load S, the reliability index has a closed-form solution:

$$\beta = \frac{\ln(\mu_R/\mu_S) - \frac{1}{2}(\sigma_{\ln R}^2 - \sigma_{\ln S}^2)}{\sqrt{\sigma_{\ln R}^2 + \sigma_{\ln S}^2}}$$


In [ ]:
# -----------------------------------------------------------------------
# Lognormal FORM (closed-form for R/S both lognormal)
# -----------------------------------------------------------------------
def lognormal_beta(mu_R, cov_R, mu_S, cov_S):
    """Reliability index β for lognormal R and S."""
    sigma_lnR = np.sqrt(np.log(1 + cov_R**2))
    sigma_lnS = np.sqrt(np.log(1 + cov_S**2))
    mu_lnR = np.log(mu_R) - 0.5 * sigma_lnR**2
    mu_lnS = np.log(mu_S) - 0.5 * sigma_lnS**2
    beta = (mu_lnR - mu_lnS) / np.sqrt(sigma_lnR**2 + sigma_lnS**2)
    return beta

beta_FORM = lognormal_beta(mu_Rc, cov_Rc, mu_S, cov_S)
Pf_FORM   = stats.norm.cdf(-beta_FORM)

print(f"FORM reliability index  β  = {beta_FORM:.3f}")
print(f"FORM failure probability Pf = {Pf_FORM:.2e}")
print(f"Target β (ULS, 50-yr)       = 4.7")
print(f"Meets target: {beta_FORM >= 4.7}")

## 5.4 Monte Carlo Simulation


In [ ]:
# -----------------------------------------------------------------------
# Monte Carlo simulation for LS1 (N = 1,000,000 samples)
# -----------------------------------------------------------------------
N_mc = 1_000_000

# Sample lognormal R and S
sigma_lnRc = np.sqrt(np.log(1 + cov_Rc**2))
mu_lnRc    = np.log(mu_Rc) - 0.5 * sigma_lnRc**2
sigma_lnS  = np.sqrt(np.log(1 + cov_S**2))
mu_lnS     = np.log(mu_S) - 0.5 * sigma_lnS**2

R_samples = rng.lognormal(mu_lnRc, sigma_lnRc, N_mc)
S_samples = rng.lognormal(mu_lnS,  sigma_lnS,  N_mc)

g_samples = R_samples - S_samples   # limit state function
Pf_MC     = np.mean(g_samples < 0)
beta_MC   = -stats.norm.ppf(Pf_MC) if Pf_MC > 0 else np.inf

print(f"MC failure probability  Pf  = {Pf_MC:.2e}  (N = {N_mc:,})")
print(f"MC reliability index    β   = {beta_MC:.3f}")
print(f"FORM β = {beta_FORM:.3f}  |  MC β = {beta_MC:.3f}  |  Difference = {abs(beta_FORM - beta_MC):.4f}")

In [ ]:
# -----------------------------------------------------------------------
# Sensitivity – β vs. mean load S (tornado plot)
# -----------------------------------------------------------------------
mu_S_range = np.linspace(400, 2500, 200)
beta_range = [lognormal_beta(mu_Rc, cov_Rc, s, cov_S) for s in mu_S_range]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(mu_S_range, beta_range, color='steelblue', linewidth=2)
ax.axhline(4.7, color='tomato', linestyle='--', label='Target β = 4.7 (ULS)')
ax.axhline(3.3, color='orange', linestyle='--', label='Target β = 3.3 (FLS)')
ax.axvline(mu_S, color='green', linestyle=':', label=f'Design load = {mu_S:.0f} kN')
ax.set_xlabel('Mean Tether Load S [kN]')
ax.set_ylabel('Reliability Index β')
ax.set_title('LS1 – Sensitivity of β to Mean Load')
ax.legend()
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 5.5 Convergence Check


In [ ]:
# -----------------------------------------------------------------------
# MC convergence plot
# -----------------------------------------------------------------------
cum_Pf = np.cumsum(g_samples < 0) / np.arange(1, N_mc + 1)
sample_idx = np.logspace(2, np.log10(N_mc), 300, dtype=int)

fig, ax = plt.subplots(figsize=(8, 4))
ax.semilogx(sample_idx, cum_Pf[sample_idx - 1], color='steelblue', linewidth=1.5)
ax.axhline(Pf_MC, color='tomato', linestyle='--', label=f'Final $P_f$ = {Pf_MC:.2e}')
ax.set_xlabel('Number of Samples')
ax.set_ylabel('Cumulative $P_f$')
ax.set_title('Monte Carlo Convergence – LS1')
ax.legend()
ax.grid(True, which='both', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 5.6 Summary Table

| Limit State | Method | β | P_f | Target β | Pass? |
|-------------|--------|---|-----|----------|-------|
| LS1 – Tether tension | FORM | *(fill)* | *(fill)* | 4.7 | *(Y/N)* |
| LS1 – Tether tension | MCS  | *(fill)* | *(fill)* | 4.7 | *(Y/N)* |
| LS2 – Fatigue | *(fill)* | *(fill)* | *(fill)* | 3.3 | *(Y/N)* |
| LS3 – Heave SLS | *(fill)* | *(fill)* | *(fill)* | 2.5 | *(Y/N)* |

---
*End of main analysis. Refer to the project report for detailed discussion.*